In [2]:
!pip install -q sentence-transformers rank-bm25 pandas numpy scikit-learn transformers torch

print("✅ Installation terminée")

✅ Installation terminée


In [3]:
import json
import pickle
import re
import time
import warnings
from pathlib import Path
from typing import List, Dict
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
print("✅ Imports réussis")

✅ Imports réussis


In [4]:
DATASET_PATH = Path("merged_cancers_vfrancais.json")

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"❌ Fichier introuvable : {DATASET_PATH}")

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ Dataset chargé : {len(data)} documents")
print(f"📊 Types de documents : {set(d.get('document_type', 'unknown') for d in data)}")

✅ Dataset chargé : 233 documents
📊 Types de documents : {'drug', 'pathology', 'oncology_score', 'resistance_mechanism', 'followup', 'oncology_emergency', 'staging_system', 'cancer_knowledge', 'metastasis', 'contraindication_interaction', 'clinical_case', 'diagnostic_guideline', 'medical_literature', 'drug_profile_standard', 'toxicity_management', 'treatment_protocol', 'biomarker_genetics', 'unknown', 'palliative_care', 'clinical_reasoning'}


In [5]:
def extract_cancer_from_docid(doc_id: str) -> str:
    """Extrait le nom du cancer à partir du document_id"""
    
    mapping = {
        'SCLC': 'Cancer du poumon à petites cellules',
        'CPNPC': 'Cancer Pulmonaire Non à Petites Cellules (CPNPC)',
        'BREAST': 'Cancer du sein',
        'CRC': 'Cancer Colorectal (CCR)',
        'PROSTATE': 'Cancer de la prostate',
        'THYROID': 'Cancer de la thyroïde',
        'OVARIAN': "Cancer épithélial de l'ovaire",
        'MEL': 'Mélanome',
        'HGG': 'Gliomes de haut grade',
        'STOMACH': 'Cancer gastrique',
    }
    
    prefix = doc_id.split('_')[0] if '_' in doc_id else doc_id
    return mapping.get(prefix, None)

# Test
test_id = "SCLC_CANCER_KNOWLEDGE"
print(f"Test: {test_id} → {extract_cancer_from_docid(test_id)}")

Test: SCLC_CANCER_KNOWLEDGE → Cancer du poumon à petites cellules


In [6]:
print("🔄 Correction des noms de cancer...")

fixed_count = 0
for doc in data:
    if not doc.get('cancer_name') or doc['cancer_name'] == 'unknown':
        cancer = extract_cancer_from_docid(doc.get('document_id', ''))
        if cancer:
            doc['cancer_name'] = cancer
            fixed_count += 1

print(f"✅ {fixed_count} documents corrigés")

# Vérification
cancer_counts = Counter(d.get('cancer_name', 'unknown') for d in data)
print("\n📊 Distribution des cancers après correction :")
for c, cnt in sorted(cancer_counts.items(), key=lambda x: -x[1])[:10]:
    if c != 'unknown':
        print(f"   {c}: {cnt}")

🔄 Correction des noms de cancer...
✅ 128 documents corrigés

📊 Distribution des cancers après correction :
   Cancer du poumon à petites cellules: 18
   Gliomes de haut grade: 18
   Cancer épithélial de l'ovaire: 18
   Cancer Pulmonaire Non à Petites Cellules (CPNPC): 18
   Cancer de la thyroïde: 17
   Cancer de la prostate: 17
   Cancer gastrique: 15
   Mélanome: 11
   Thyroid Cancer: 1


In [7]:
# Normalisation des noms anglais vers français
normalization = {
    'Thyroid Cancer': 'Cancer de la thyroïde',
    'Prostate Cancer': 'Cancer de la prostate',
    'Kidney Cancer (Renal Cell Carcinoma)': 'Cancer du rein',
    'Gastric Cancer (Gastric Adenocarcinoma)': 'Cancer gastrique',
}

for doc in data:
    old_name = doc.get('cancer_name', '')
    if old_name in normalization:
        doc['cancer_name'] = normalization[old_name]
        print(f"   Normalisé: {old_name} → {normalization[old_name]}")

# Vérification finale
cancer_counts = Counter(d.get('cancer_name', 'unknown') for d in data)
print("\n📊 Distribution finale des cancers :")
for c, cnt in sorted(cancer_counts.items(), key=lambda x: -x[1]):
    if c != 'unknown':
        print(f"   {c}: {cnt}")

print(f"\n✅ Total documents: {len(data)}")
print(f"✅ Cancers identifiés: {len([c for c in cancer_counts if c != 'unknown'])}")

   Normalisé: Thyroid Cancer → Cancer de la thyroïde
   Normalisé: Kidney Cancer (Renal Cell Carcinoma) → Cancer du rein
   Normalisé: Prostate Cancer → Cancer de la prostate
   Normalisé: Gastric Cancer (Gastric Adenocarcinoma) → Cancer gastrique

📊 Distribution finale des cancers :
   Cancer du poumon à petites cellules: 18
   Gliomes de haut grade: 18
   Cancer épithélial de l'ovaire: 18
   Cancer de la thyroïde: 18
   Cancer de la prostate: 18
   Cancer Pulmonaire Non à Petites Cellules (CPNPC): 18
   Cancer gastrique: 16
   Mélanome: 11
   Cancer du rein: 1
   Cancer du col de l'utérus: 1
   Cancer de la vessie: 1
   Mélanome uvéal (oculaire): 1
   Carcinome urothélial in situ de la vessie: 1
   Cancer Colorectal (CCR): 1
   Cancer du sein: 1

✅ Total documents: 233
✅ Cancers identifiés: 15


In [8]:
def chunk_document(item: Dict) -> List[Dict]:
    """Découpe un document en chunks pour le RAG"""
    chunks = []
    cancer = item.get('cancer_name', 'unknown')
    doc_type = item.get('document_type', 'unknown')
    doc_id = item.get('document_id', 'unknown')
    
    def add_chunk(text: str, subtype: str):
        if len(text) > 50:
            chunks.append({
                'text': f"[{cancer}] {subtype}: {text[:500]}",
                'metadata': {
                    'cancer': cancer,
                    'type': doc_type,
                    'subtype': subtype,
                    'doc_id': doc_id
                }
            })
    
    # Traitement selon le type de document
    if doc_type == 'treatment_protocol':
        for p in item.get('protocols', []):
            text = f"Protocole: {p.get('protocol_name', 'N/A')} | Médicaments: {p.get('drugs', [])}"
            add_chunk(text, 'protocol')
    
    elif doc_type == 'cancer_knowledge':
        for key in ['definition', 'diagnosis', 'treatment_overview']:
            if key in item:
                add_chunk(str(item[key]), key)
    
    elif doc_type == 'clinical_case':
        for case in item.get('clinical_cases', []):
            text = f"Cas: {case.get('title', '')} | Diagnostic: {case.get('diagnosis', '')}"
            add_chunk(text, 'case')
    
    elif doc_type == 'drug':
        for drug in item.get('drugs', []):
            text = f"Médicament: {drug.get('name', 'N/A')} | Classe: {drug.get('drug_class', '')}"
            add_chunk(text, 'drug')
    
    else:
        # Autres types: chunk générique
        full = json.dumps(item, ensure_ascii=False)[:500]
        if len(full) > 100:
            add_chunk(full, doc_type)
    
    return chunks

print("✅ Fonction de chunking définie")

✅ Fonction de chunking définie


In [9]:
print("🔄 Création des chunks...")
all_chunks = []
for doc in data:
    all_chunks.extend(chunk_document(doc))

print(f"✅ {len(all_chunks)} chunks créés")

# Statistiques
chunk_types = Counter(c['metadata']['type'] for c in all_chunks)
print("\n📊 Types de chunks :")
for t, cnt in chunk_types.most_common(10):
    print(f"   {t}: {cnt}")

# Sauvegarde
with open('chunks.json', 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print("\n💾 Chunks sauvegardés: chunks.json")

🔄 Création des chunks...
✅ 250 chunks créés

📊 Types de chunks :
   treatment_protocol: 51
   cancer_knowledge: 14
   metastasis: 13
   resistance_mechanism: 13
   staging_system: 13
   oncology_score: 13
   followup: 13
   contraindication_interaction: 13
   toxicity_management: 13
   palliative_care: 13

💾 Chunks sauvegardés: chunks.json


In [10]:
from sentence_transformers import SentenceTransformer

print("🔄 Chargement du modèle SBERT (multilingue)...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"✅ Modèle chargé (dimension: {model.get_sentence_embedding_dimension()})")

🔄 Chargement du modèle SBERT (multilingue)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8265.91it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé (dimension: 384)


In [11]:
# Préparer les textes
texts = [c['text'] for c in all_chunks]
print(f"📝 {len(texts)} textes à encoder")

# Calcul des embeddings
print("🔄 Calcul des embeddings (cela peut prendre 1-2 minutes)...")
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print(f"✅ Embeddings calculés : {embeddings.shape}")

# Sauvegarde
with open('embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)
print("💾 Embeddings sauvegardés: embeddings.pkl")

📝 250 textes à encoder
🔄 Calcul des embeddings (cela peut prendre 1-2 minutes)...


Batches: 100%|██████████| 8/8 [00:23<00:00,  2.89s/it]

✅ Embeddings calculés : (250, 384)
💾 Embeddings sauvegardés: embeddings.pkl


In [12]:
from rank_bm25 import BM25Okapi
import re
import numpy as np
from typing import List, Dict

class HybridRetriever:
    """
    Retriever hybride : BM25 (recherche lexicale) + SBERT (recherche sémantique)
    """
    
    # Boost par type de document
    DOC_TYPE_BOOST = {
        'treatment_protocol': 3.0,
        'clinical_case': 1.8,
        'cancer_knowledge': 1.2,
        'diagnostic_guideline': 1.3,
        'clinical_reasoning': 1.2,
        'drug': 2.0,
        'followup': 0.9,
        'default': 1.0
    }
    
    def __init__(self, chunks, embeddings, model, bm25_weight=0.4, semantic_weight=0.6):
        self.chunks = chunks
        self.embeddings = embeddings
        self.model = model
        self.bm25_weight = bm25_weight
        self.semantic_weight = semantic_weight
        
        # Création de l'index BM25
        print("🔄 Création de l'index BM25...")
        texts = [c['text'] for c in chunks]
        tokenized = [self._tokenize(t) for t in texts]
        self.bm25 = BM25Okapi(tokenized)
        print(f"✅ Index BM25 créé avec {len(tokenized)} documents")
    
    def _tokenize(self, text: str) -> List[str]:
        """Tokenisation pour BM25"""
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        return text.split()
    
    def _get_boost(self, doc_type: str) -> float:
        return self.DOC_TYPE_BOOST.get(doc_type, self.DOC_TYPE_BOOST['default'])
    
    def _normalize(self, scores: np.ndarray) -> np.ndarray:
        if len(scores) == 0 or np.max(scores) == 0:
            return scores
        return scores / np.max(scores)
    
    def search(self, query: str, k: int = 5, cancer_filter: str = None) -> List[Dict]:
        # 1. BM25 (recherche lexicale)
        tokens = self._tokenize(query)
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_indices = np.argsort(bm25_scores)[-k*3:][::-1]
        bm25_norm = self._normalize(bm25_scores[bm25_indices])
        
        # 2. SBERT (recherche sémantique)
        q_emb = self.model.encode([query])
        semantic_scores = np.dot(self.embeddings, q_emb.T).flatten()
        semantic_indices = np.argsort(semantic_scores)[-k*3:][::-1]
        semantic_norm = self._normalize(semantic_scores[semantic_indices])
        
        # 3. Fusion des scores
        combined = {}
        for idx, sc in zip(bm25_indices, bm25_norm):
            combined[idx] = self.bm25_weight * sc
        for idx, sc in zip(semantic_indices, semantic_norm):
            combined[idx] = combined.get(idx, 0) + self.semantic_weight * sc
        
        # 4. Construction des résultats avec boost et filtre
        results = []
        for idx, score in sorted(combined.items(), key=lambda x: x[1], reverse=True):
            chunk = self.chunks[idx]
            cancer = chunk['metadata'].get('cancer', 'unknown')
            doc_type = chunk['metadata'].get('type', 'unknown')
            
            # Filtre par cancer
            if cancer_filter and cancer_filter.lower() not in cancer.lower():
                continue
            
            final_score = score * self._get_boost(doc_type)
            
            results.append({
                'text': chunk['text'],
                'score': final_score,
                'bm25_score': bm25_scores[idx] if idx in bm25_indices else 0,
                'semantic_score': semantic_scores[idx] if idx in semantic_indices else 0,
                'cancer': cancer,
                'type': doc_type,
            })
            
            if len(results) >= k:
                break
        
        return results
    
    def search_strict(self, query: str, k: int = 5, cancer_filter: str = None) -> List[Dict]:
        """Version avec filtre strict sur le cancer (correspondance exacte)"""
        q_emb = self.model.encode([query])
        scores = np.dot(self.embeddings, q_emb.T).flatten()
        
        # BM25
        tokens = self._tokenize(query)
        bm25_scores = self.bm25.get_scores(tokens)
        
        # Fusion avec filtre strict
        results = []
        for idx, score in enumerate(scores):
            chunk = self.chunks[idx]
            cancer = chunk['metadata'].get('cancer', 'unknown')
            doc_type = chunk['metadata'].get('type', 'unknown')
            
            # FILTRE STRICT : correspondance exacte
            if cancer_filter:
                if cancer_filter.lower() != cancer.lower():
                    continue
            
            # Score hybride
            bm25_score = bm25_scores[idx]
            hybrid_score = (self.bm25_weight * bm25_score + self.semantic_weight * score) / (self.bm25_weight + self.semantic_weight)
            final_score = hybrid_score * self._get_boost(doc_type)
            
            results.append({
                'text': chunk['text'],
                'score': final_score,
                'cancer': cancer,
                'type': doc_type,
            })
        
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:k]
    
    def search_with_auto_detect(self, query: str, k: int = 5) -> List[Dict]:
        """Recherche avec détection automatique du cancer"""
        cancers = ["Cancer de la prostate", "Cancer du sein", "Cancer gastrique",
                   "Cancer du poumon", "Cancer de l'ovaire", "Cancer de la thyroïde",
                   "Gliomes de haut grade", "Mélanome", "Cancer colorectal"]
        
        detected_cancer = None
        query_lower = query.lower()
        for cancer in cancers:
            if cancer.lower() in query_lower:
                detected_cancer = cancer
                break
        
        return self.search_strict(query, k=k, cancer_filter=detected_cancer)

# Initialisation
retriever = HybridRetriever(all_chunks, embeddings, model, bm25_weight=0.4, semantic_weight=0.6)
print("\n✅ Retriever hybride avec search_strict prêt!")

🔄 Création de l'index BM25...
✅ Index BM25 créé avec 250 documents

✅ Retriever hybride avec search_strict prêt!


In [13]:
# Utiliser le retriever que vous avez déjà défini
# (pas besoin de recréer une classe)

# Tester avec un cancer qui a des protocoles
test_cancer = "Cancer de la prostate"

print("="*60)
print(f"🔍 TEST AVEC: {test_cancer}")
print("="*60)

query = f"Traitement du {test_cancer}"
print(f"❓ {query}\n")

results = retriever.search(query, k=4, cancer_filter=test_cancer)

for i, r in enumerate(results, 1):
    print(f"\n{i}. [Score: {r['score']:.4f}]")
    print(f"   Type: {r['type']}")
    print(f"   Extrait: {r['text'][:200]}...")
    print("-"*50)

🔍 TEST AVEC: Cancer de la prostate
❓ Traitement du Cancer de la prostate


1. [Score: 2.8651]
   Type: treatment_protocol
   Extrait: [Cancer de la prostate] protocol: Protocole: Very Low-Risk and Low-Risk Localized Prostate Cancer — Active Surveillance | Médicaments: []...
--------------------------------------------------

2. [Score: 1.7062]
   Type: clinical_case
   Extrait: [Cancer de la prostate] case: Cas: PSA Elevation in a 62-Year-Old Man — Navigating Biopsy Decision | Diagnostic: Stratification du risque de cancer de la prostate cliniquement significatif ; lésion PI...
--------------------------------------------------

3. [Score: 1.0748]
   Type: cancer_knowledge
   Extrait: [Cancer de la prostate] definition: Le cancer de la prostate est une tumeur épithéliale maligne provenant des cellules glandulaires de la prostate. Il constitue un spectre de maladies biologiquement h...
--------------------------------------------------

4. [Score: 1.5714]
   Type: clinical_case
   Extra

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

class Phi3Generator:

    def __init__(self):
        self.model_name = "microsoft/Phi-3-mini-4k-instruct"
        self.tokenizer = None
        self.model = None

    def load(self):

        print("🔄 Chargement Phi-3 Mini...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            trust_remote_code=True,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )

        self.model.eval()

        print("✅ Phi-3 chargé")

    def generate(self, prompt, max_new_tokens=300):

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        )

        with torch.no_grad():

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        return response

In [15]:
def build_prompt(query: str, context: str) -> str:
    return f"""<|im_start|>system
Tu es un assistant médical spécialisé en oncologie. Réponds à la question en utilisant UNIQUEMENT le contexte fourni. Structure ta réponse en 3 parties: 1) Traitement, 2) Effets secondaires, 3) Suivi.
<|im_end|>
<|im_start|>user
CONTEXTE:
{context}

QUESTION: {query}

RÉPONSE STRUCTURÉE:
<|im_end|>
<|im_start|>assistant
"""

def rag_answer(query: str, k: int = 3, cancer_filter: str = None) -> Dict:
    start_time = time.time()
    
    # 1. Retrieval
    results = retriever.search(query, k=k, cancer_filter=cancer_filter)
    
    # 2. Construction du contexte
    context = "\n\n".join([f"Source {i+1}: {r['text']}" for i, r in enumerate(results)])
    
    # 3. Génération
    prompt = build_prompt(query, context)
    answer = llm.generate(prompt, max_new_tokens=300)
    
    return {
        'query': query,
        'answer': answer,
        'sources': [(r['cancer'], r['type'], r['score']) for r in results],
        'time': time.time() - start_time
    }

In [18]:
# Fonction de détection automatique du cancer dans la question
def detect_cancer_in_question(question: str) -> str:
    """Détecte automatiquement le type de cancer dans la question"""
    cancers = [
        "Cancer de la prostate",
        "Cancer du sein", 
        "Cancer gastrique",
        "Cancer du poumon",
        "Cancer de l'ovaire",
        "Cancer de la thyroïde",
        "Gliomes de haut grade",
        "Mélanome",
        "Cancer colorectal",
        "Cancer du rein",
        "Cancer de la vessie",
    ]
    
    question_lower = question.lower()
    for cancer in cancers:
        if cancer.lower() in question_lower:
            return cancer
    return None

# Test avec différentes questions
test_questions = [
    "Quel est le traitement du cancer de la prostate ?",
    "Suivi après un cancer du sein",
    "Diagnostic du cancer gastrique précoce",
    "Effets secondaires chimiothérapie cancer du poumon",
    "Pronostic du mélanome",
    "Traitement du cancer colorectal métastatique",
]

print("="*60)
print("🔍 TEST DE DÉTECTION AUTOMATIQUE")
print("="*60)

for q in test_questions:
    cancer = detect_cancer_in_question(q)
    print(f"\n❓ {q}")
    print(f"🎯 Cancer détecté: {cancer if cancer else 'Non détecté'}")
    
    if cancer:
        results = retriever.search(q, k=2, cancer_filter=cancer)
        if results:
            print(f"   ✅ {len(results)} résultat(s) trouvé(s)")
            for r in results:
                print(f"      - {r['type']} (score: {r['score']:.2f})")
        else:
            print(f"   ❌ Aucun résultat pour {cancer}")

🔍 TEST DE DÉTECTION AUTOMATIQUE

❓ Quel est le traitement du cancer de la prostate ?
🎯 Cancer détecté: Cancer de la prostate
   ✅ 2 résultat(s) trouvé(s)
      - treatment_protocol (score: 1.80)
      - clinical_case (score: 0.85)

❓ Suivi après un cancer du sein
🎯 Cancer détecté: Cancer du sein
   ✅ 1 résultat(s) trouvé(s)
      - cancer_knowledge (score: 0.65)

❓ Diagnostic du cancer gastrique précoce
🎯 Cancer détecté: Cancer gastrique
   ✅ 2 résultat(s) trouvé(s)
      - treatment_protocol (score: 1.80)
      - clinical_reasoning (score: 0.63)

❓ Effets secondaires chimiothérapie cancer du poumon
🎯 Cancer détecté: Cancer du poumon
   ✅ 2 résultat(s) trouvé(s)
      - treatment_protocol (score: 3.00)
      - treatment_protocol (score: 1.57)

❓ Pronostic du mélanome
🎯 Cancer détecté: Mélanome
   ✅ 2 résultat(s) trouvé(s)
      - cancer_knowledge (score: 1.11)
      - toxicity_management (score: 0.53)

❓ Traitement du cancer colorectal métastatique
🎯 Cancer détecté: Cancer colorectal
 

In [19]:
# Tester sur les cancers avec beaucoup de données
print("="*70)
print("🏥 TEST SUR LES CANCERS AVEC BEAUCOUP DE DONNÉES")
print("="*70)

cancers_bien_representes = [
    ("Cancer de la prostate", "Quel est le traitement du cancer de la prostate à un stade avancé ?"),
    ("Cancer gastrique", "Quel est le pronostic du cancer gastrique ?"),
    ("Cancer du poumon à petites cellules", "Traitement du cancer du poumon à petites cellules"),
    ("Cancer de l'ovaire", "Comment diagnostiquer le cancer de l'ovaire ?"),
    ("Gliomes de haut grade", "Quel est le traitement des gliomes de haut grade ?"),
    ("Cancer de la thyroïde", "Quels sont les facteurs de risque du cancer de la thyroïde ?"),
]

for cancer, question in cancers_bien_representes:
    print(f"\n{'='*60}")
    print(f"❓ {question}")
    print(f"🎯 Cancer ciblé: {cancer}")
    print('='*60)
    
    results = retriever.search(question, k=2, cancer_filter=cancer)
    
    if results:
        for i, r in enumerate(results, 1):
            print(f"\n{i}. [Score: {r['score']:.2f}] Type: {r['type']}")
            # Afficher un extrait plus long pour mieux juger
            print(f"   {r['text'][:300]}...")
            print()
    else:
        print("❌ Aucun résultat trouvé")

🏥 TEST SUR LES CANCERS AVEC BEAUCOUP DE DONNÉES

❓ Quel est le traitement du cancer de la prostate à un stade avancé ?
🎯 Cancer ciblé: Cancer de la prostate

1. [Score: 1.40] Type: clinical_case
   [Cancer de la prostate] case: Cas: High-Risk Prostate Cancer with Lymph Node Involvement — Multimodal Treatment Planning | Diagnostic: Cancer de la prostate localement avancé à très haut risque : cT3b N1 M0, ISUP GG 5, PSA 45 ng/mL...


2. [Score: 1.80] Type: treatment_protocol
   [Cancer de la prostate] protocol: Protocole: Very Low-Risk and Low-Risk Localized Prostate Cancer — Active Surveillance | Médicaments: []...


❓ Quel est le pronostic du cancer gastrique ?
🎯 Cancer ciblé: Cancer gastrique

1. [Score: 0.72] Type: clinical_reasoning
   [Cancer gastrique] clinical_reasoning: {"document_id": "STOMACH_CLINICAL_REASONING", "document_type": "clinical_reasoning", "created_by": "Aya_Hajar_Saloua_Douae_Ayoub", "last_updated": "2026-06-06", "evidence_level": "Élevé", "references": [{"source":